# Demo: Building a Finance Q&A Agent using LangGraph

### Step 0: Install Required Libraries
<details>
<summary>📘 Explanation</summary>

- `langchain`: The core framework used to build chains and agents that connect LLMs with tools.
- `langchain-openai`: A specific integration that enables access to OpenAI and Azure OpenAI models.
</details>

In [ ]:
# Install LangChain and OpenAI integrations
%pip install langchain langchain-openai

### Step 1: Import Required Libraries and Set Up LLM

In [1]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
# Load environment variables from .env file
load_dotenv()

llm = ChatOpenAI(    
    model="gpt-5-mini",
    temperature=0,
    api_key=os.environ["OPENAI_API_KEY"]
)

def safe_python_repl(code: str) -> str:
    try:
        local_vars = {}
        exec(code, {}, local_vars)
        return str(local_vars)
    except Exception as e:
        return f"Error: {str(e)}"

class PythonREPL:
    def run(self, code):
        return safe_python_repl(code)

repl = PythonREPL()


### Step 2: Define the Prompt Template

In [2]:
prompt = PromptTemplate.from_template(
    """You are a smart assistant for product managers. 
Given a business-related question, return:
1. Valid Python code using print(...) to calculate the answer.
2. A brief explanation of the result in plain English.

Question: {input}
Response:"""
)

### Step 3: Define the Helper Function

In [3]:
def process_pm_query(q):
    response = llm.invoke(prompt.format(input=q)).content.strip()

    if "```python" in response:
        code_block = response.split("```python")[1].split("```")[0].strip()
    else:
        code_block = next((line for line in response.splitlines() if "print" in line), "")

    try:
        result = repl.run(code_block)
        return {
            "question": q,
            "code": code_block,
            "result": result.strip(),
            "explanation": response.replace(code_block, "").replace("```python", "").replace("```", "").strip()
        }
    except Exception as e:
        return {
            "question": q,
            "code": code_block,
            "error": str(e),
            "raw_response": response
        }

### Step 4: Run the Demo

In [4]:
# --- Questions for PMs ---

questions = [
    "If we grow 8% monthly, what is our user count after 6 months starting from 10,000?",
    "We lose 25% of our 40,000 users. How many do we retain?",
    "If each user pays $10/month and we have 5000 users, what is the monthly revenue?",
    "If revenue is $50,000 and cost is $37,000, what’s our profit?"
]

for q in questions:
    print(f"\nQuestion: {q}")
    output = process_pm_query(q)
    print("Code:", output.get("code", "No code"))
    print("Result:", output.get("result", output.get("error", "Failed to execute")))
    print("Explanation:", output.get("explanation", "No explanation"))


Question: If we grow 8% monthly, what is our user count after 6 months starting from 10,000?
Code: print(f"Users after {months} months: {final:,.0f}")
Result: Error: name 'months' is not defined
Explanation: initial = 10000
monthly_growth = 0.08
months = 6
final = initial * (1 + monthly_growth) ** months


Explanation:
Starting from 10,000 users, 8% monthly compound growth for 6 months yields about 10,000 * 1.08^6 ≈ 15,869 users.

Question: We lose 25% of our 40,000 users. How many do we retain?
30000
Code: print(int(40000 * (1 - 0.25)))
Result: {}
Explanation: You lose 25% of 40,000, so you retain 75%: 40,000 * 0.75 = 30,000 users (10,000 users lost).

Question: If each user pays $10/month and we have 5000 users, what is the monthly revenue?
Code: print(f"${monthly_revenue:,}")
Result: Error: name 'monthly_revenue' is not defined
Explanation: # Python calculation
price_per_user = 10
users = 5000
monthly_revenue = price_per_user * users


Explanation: With each user paying $10/month a